In [5]:
def buscar_cancion_id(nombre, dataset, artista=None):
    """
    Busca el track_id de una canción por nombre (y artista opcional).
    """
    # Busqueda insensible a mayúsculas
    if artista:
        resultado = dataset[(dataset['track_name'].str.lower() == nombre.lower()) & 
                            (dataset['artists'].str.lower() == artista.lower())]
    else:
        resultado = dataset[dataset['track_name'].str.lower() == nombre.lower()]
    
    if resultado.empty:
        print(f"❌ No se encontró la canción: '{nombre}'")
        return None
    
    # Si hay varios, tomamos el primero y avisamos
    if len(resultado) > 1 and not artista:
        print(f"⚠️ Se encontraron varias versiones de '{nombre}'. Usando la de {resultado.iloc[0]['artists']}.")
    
    return resultado.iloc[0]['track_id']

In [4]:
# ==========================================
# EXPERIMENTO COMPARATIVO: DBSCAN
# Evaluación de KPIs contra el modelo base
# ==========================================

import pandas as pd
import time
import sys
from sklearn.cluster import DBSCAN

# 1. Importamos el evaluador de métricas que creó el equipo
# Asegúrate de que el notebook se esté ejecutando en la carpeta correcta
sys.path.append('../src') 
try:
    from evaluador import evaluar_modelo_recomendacion
except ImportError:
    print("⚠️ No se encontró evaluador.py. Asegúrate de tener la carpeta src/ correctamente configurada.")

print("--- 1. Cargando Datos ---")
# Usa una ruta relativa para que funcione en cualquier computadora del equipo
ruta_csv = "C:/Users/basti/proyecto_cd/data/processed/spotify_dataset_processed.csv"

try:
    df = pd.read_csv(ruta_csv)
    print(f"✅ Dataset cargado. Dimensiones: {df.shape}")
except FileNotFoundError:
    raise SystemExit("❌ Error: No se encontró el dataset procesado.")

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# --- MODELO 2: DBSCAN (El retador) ---
print("\n⏳ Entrenando DBSCAN (Calculando densidades globales)...")
# Usamos los parámetros definidos por tu compañero
modelo_dbscan = DBSCAN(eps=0.25, min_samples=10, metric='euclidean', n_jobs=-1)
df['dbscan_cluster'] = modelo_dbscan.fit_predict(df[features])

n_ruido = list(df['dbscan_cluster'].values).count(-1)
print(f"✅ Modelo entrenado. (Nota Crítica: Clasificó {n_ruido} canciones como 'Ruido').")
print("==========================================\n")

# --- 3. FUNCIÓN DE EXTRACCIÓN Y EVALUACIÓN DBSCAN ---
def probar_dbscan_y_evaluar(track_id, dataset, features_list, top_k=5):
    inicio_tiempo = time.time()
    
    cancion_semilla = dataset[dataset['track_id'] == track_id]
    if cancion_semilla.empty:
        return "Canción no encontrada."

    nombre = cancion_semilla['track_name'].values[0]
    artista = cancion_semilla['artists'].values[0]
    vector_semilla = cancion_semilla[features_list]
    cluster_id = cancion_semilla['dbscan_cluster'].values[0]

    print(f"🎵 CANCIÓN SEMILLA: '{nombre}' de {artista}")
    print("-" * 50)

    # Escenario A: La canción es considerada RUIDO (-1)
    if cluster_id == -1:
        fin_tiempo = time.time()
        print(f"❌ FALLO CRÍTICO: DBSCAN dice que esta canción es RUIDO (-1).")
        print("El sistema no puede recomendar nada porque la pista no pertenece a ningún grupo.")
        # Llamamos al evaluador enviando un dataframe vacío para registrar el fallo
        evaluar_modelo_recomendacion("DBSCAN (Ruido)", vector_semilla, pd.DataFrame(), inicio_tiempo, fin_tiempo)
        return None

    # Escenario B: La canción pertenece a un clúster
    print(f"DBSCAN asignó la canción al Clúster {cluster_id}.")
    
    # Extraemos todas las canciones de ese mismo clúster (excluyendo la semilla)
    canciones_mismo_cluster = dataset[(dataset['dbscan_cluster'] == cluster_id) & (dataset['track_id'] != track_id)]
    
    # Como DBSCAN no ordena por distancia internamente, tomamos una muestra aleatoria del clúster
    if len(canciones_mismo_cluster) >= top_k:
        recomendadas = canciones_mismo_cluster.sample(n=top_k, random_state=42)
    else:
        recomendadas = canciones_mismo_cluster # Si hay menos de 5, tomamos las que haya
        
    vectores_recomendados = recomendadas[features_list]
    fin_tiempo = time.time()
    
    # 4. ENVIAMOS LOS VECTORES AL EVALUADOR PARA CALCULAR LOS KPIs
    evaluar_modelo_recomendacion("DBSCAN", vector_semilla, vectores_recomendados, inicio_tiempo, fin_tiempo)
    
    return recomendadas

# --- 4. EJECUCIÓN DE LA PRUEBA ---
# Es VITAL usar exactamente el mismo track_id que usaste en tu notebook de KNN
test_track_id = df['track_id'].iloc[1000] 
resultados_dbscan = probar_dbscan_y_evaluar(test_track_id, df, features)

--- 1. Cargando Datos ---
✅ Dataset cargado. Dimensiones: (81343, 20)

⏳ Entrenando DBSCAN (Calculando densidades globales)...
✅ Modelo entrenado. (Nota Crítica: Clasificó 1350 canciones como 'Ruido').

🎵 CANCIÓN SEMILLA: 'Me Dê Motivo' de Criolo
--------------------------------------------------
DBSCAN asignó la canción al Clúster 0.

=== EVALUACIÓN DEL MODELO: DBSCAN ===
⏱️ KPI Latencia: 0.0245 segundos
📏 KPI Distancia Acústica Promedio: 0.1309
⚠️ Resultado: FALLO (Recomendaciones demasiado dispersas acústicamente)

